In [0]:
# Paths
GOLD_PATH = "/Volumes/workspace/default/logs_volume/gold"
SILVER_PATH = "/Volumes/workspace/default/logs_volume/silver"
BRONZE_PATH = "/Volumes/workspace/default/logs_volume/bronze"
STREAM_INPUT  = "/Volumes/workspace/default/logs_volume/input"   # we'll drop files here
STREAM_OUTPUT = "/Volumes/workspace/default/logs_volume/output"
STREAM_CHKPT = "/Volumes/workspace/default/logs_volume/checkpoints"


In [0]:
# Load all Gold tables computed by batch notebooks
ip_baselines    = spark.read.format("delta").load(f"{GOLD_PATH}/ip_behaviour")
hourly_baseline = spark.read.format("delta").load(f"{GOLD_PATH}/hourly_stats")
endpoint_ref    = spark.read.format("delta").load(f"{GOLD_PATH}/endpoint_stats")

# Pull the global hourly mean and stddev batch already computed
from pyspark.sql.functions import avg, stddev
global_stats = hourly_baseline.agg(
    avg("total_requests").alias("mean"),
    stddev("total_requests").alias("stddev")
).collect()[0]

GLOBAL_MEAN   = global_stats["mean"]
GLOBAL_STDDEV = global_stats["stddev"]

print(f"Batch baseline loaded.")
print(f"Global hourly mean: {GLOBAL_MEAN:.0f}, stddev: {GLOBAL_STDDEV:.0f}")
print(f"Known IPs in baseline: {ip_baselines.count():,}")
print(f"Known endpoints:       {endpoint_ref.count():,}")

Batch baseline loaded.
Global hourly mean: 2850, stddev: 1679
Known IPs in baseline: 81,892
Known endpoints:       21,079


In [0]:
from pyspark.sql.functions import col, round

# Streaming only needs per-IP mean and stddev to score live requests
# Selecting minimal columns keeps the join lightweight
ip_profile = (ip_baselines
    .select(
        col("host"),
        col("total_requests").alias("batch_total_requests"),
        col("error_rate_pct").alias("batch_error_rate"),
        col("unique_endpoints_hit").alias("batch_endpoint_breadth"),
        # Derive a "normal hourly rate" from total requests across ~28 days
        round(col("total_requests") / (28 * 24), 2).alias("expected_hourly_requests")
    )
)

# Known broken endpoints from batch
broken_endpoints = (endpoint_ref
    .filter(
        round(col("not_found_count") / col("total_hits"), 2) > 0.2
    )
    .select("endpoint")
)

print(f"IP profiles ready:       {ip_profile.count():,}")
print(f"Known broken endpoints:  {broken_endpoints.count():,}")

IP profiles ready:       81,892
Known broken endpoints:  2,270


In [0]:
from pyspark.sql.types import StructType, StructField, StringType

raw_schema = StructType([StructField("value", StringType(), True)])

dbutils.fs.mkdirs(STREAM_INPUT)
dbutils.fs.mkdirs(STREAM_CHKPT)

stream_raw = (spark
    .readStream
    .format("text")
    .schema(raw_schema)
    .option("maxFilesPerTrigger", 1)
    .load(STREAM_INPUT)
)

print(f"Stream reader defined. isStreaming: {stream_raw.isStreaming}")

Stream reader defined. isStreaming: True


In [0]:
# This is identical to 02_silver_parsing — not rewritten, just reapplied
# In a real project this would be a shared function imported from a module
from pyspark.sql.functions import regexp_extract, when, try_to_timestamp, lit

# Each group in the pattern captures one field
LOG_PATTERN = r'^(\S+) \S+ \S+ \[([^\]]+)\] "(\S+) (\S+) (\S+)" (\d{3}) (\S+)'

parsed_df = stream_raw.select(
    regexp_extract("value", LOG_PATTERN, 1).alias("host"),
    regexp_extract("value", LOG_PATTERN, 2).alias("timestamp_raw"),
    regexp_extract("value", LOG_PATTERN, 3).alias("method"),
    regexp_extract("value", LOG_PATTERN, 4).alias("endpoint"),
    regexp_extract("value", LOG_PATTERN, 5).alias("protocol"),
    regexp_extract("value", LOG_PATTERN, 6).alias("status_code_raw"),
    regexp_extract("value", LOG_PATTERN, 7).alias("bytes_raw")
)

stream_parsed = parsed_df.select(
    col("host"),

    try_to_timestamp(
        when(col("timestamp_raw") == "", lit(None))
        .otherwise(col("timestamp_raw")),
        lit("dd/MMM/yyyy:HH:mm:ss Z")
    ).alias("timestamp"),

    col("method"),
    col("endpoint"),
    col("protocol"),

    when(col("status_code_raw") == "", lit(None))
    .otherwise(col("status_code_raw"))
    .cast("int")
    .alias("status"),

    when((col("bytes_raw") == "-") | (col("bytes_raw") == ""), 0)
    .otherwise(col("bytes_raw"))
    .cast("int")
    .alias("bytes")
).filter((col("host") != "") & (col("host").isNotNull()))


In [0]:
from pyspark.sql.functions import window, count
# Count what each IP is doing in each 10-minute window
live_ip_activity = (stream_parsed
    .withWatermark("timestamp", "10 minutes")
    .groupBy(
        window(col("timestamp"), "10 minutes", "5 minutes"),
        col("host")
    )
    .agg(
        count("*").alias("live_requests"),
        count(when(col("status") >= 400, 1)).alias("live_errors"),
        count(when(col("status") == 404, 1)).alias("live_404s")
    )
    .select(
        col("host"),
        col("window.start").alias("window_start"),
        col("live_requests"),
        col("live_errors"),
        col("live_404s"),
        round(col("live_errors") / col("live_requests") * 100, 2).alias("live_error_rate")
    )
)

In [0]:
from pyspark.sql.functions import broadcast, coalesce
# Join live activity against what batch told us about each IP
stream_enriched = (live_ip_activity
    .join(
        broadcast(ip_profile),  # broadcast = small table, don't shuffle it
        on="host",
        how="left"              # left = keep IPs we've never seen before too
    )
    .withColumn(
        # How many times their normal hourly rate are they currently running?
        "rate_multiplier",
        round(col("live_requests")/(coalesce(col("expected_hourly_requests"), lit(0)) / 6 + lit(1)),2)
        # /6 because our window is 10 min = 1/6 of an hour
        # +1 avoids division by zero for new IPs
    )
    .withColumn(
        # Z-score against GLOBAL baseline for IPs we've never seen
        "global_z_score",
        round((col("live_requests") - lit(GLOBAL_MEAN / 6))
              / lit(GLOBAL_STDDEV / 6), 2)
    )
)

In [0]:
stream_anomalies = (
    stream_enriched
    .withColumn(
        "anomaly_type",
        
        # Strong known-identity behavioral spike
        when(
            (col("batch_total_requests").isNotNull()) &
            (col("rate_multiplier") > 10),
            "PERSONAL_SPIKE"
        )

        # Known IP deviating moderately but sustained
        .when(
            (col("batch_total_requests").isNotNull()) &
            (col("rate_multiplier") > 5),
            "SUSPICIOUS_ACTIVITY"
        )

        # Unknown IP behaving abnormally
        .when(
            (col("batch_total_requests").isNull()) &
            (col("global_z_score") > 3),
            "UNKNOWN_IP_SURGE"
        )

        # Error-heavy behavior (scanner / broken client)
        .when(
            col("live_error_rate") > 30,
            "LIVE_ERROR_SURGE"
        )

        # Endpoint probing behavior
        .when(
            (col("live_404s") > 20) &
            (col("batch_error_rate") < 5),
            "SCANNING_BEHAVIOUR"
        )

        .otherwise(None)
    )
    .filter(col("anomaly_type").isNotNull())

    .withColumn(
        "severity",
        when(
            (col("rate_multiplier") > 15) | (col("live_error_rate") > 60),
            "CRITICAL"
        )
        .when(
            (col("rate_multiplier") > 8) | (col("global_z_score") > 4),
            "HIGH"
        )
        .when(
            (col("rate_multiplier") > 5) | (col("live_error_rate") > 30),
            "MEDIUM"
        )
        .otherwise("LOW")
    )
)

In [0]:
from pyspark.sql.functions import col

# Read BRONZE (already raw logs in 'value' column)
bronze_df = spark.read.format("delta").load(BRONZE_PATH)

# Ensure only raw text column exists
log_text_df = bronze_df.select(col("value"))

# Write as multiple files (simulating streaming ingestion)
(log_text_df
    .repartition(20)
    .write
    .format("text")
    .mode("overwrite")
    .save(STREAM_INPUT)
)

print("Stream input seeded from BRONZE as raw text logs")

# Start the stream
stream_query = (stream_anomalies
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", STREAM_CHKPT)
    .trigger(availableNow=True)
    .start(STREAM_OUTPUT)
)

print(f"Stream running. ID: {stream_query.id}")


Stream input seeded from BRONZE as raw text logs
Stream running. ID: f1326e13-97d8-434a-85e4-43fcdecd64b4


In [0]:
import time

print("Monitoring stream...\n")

for i in range(6):
    time.sleep(10)
    progress = stream_query.lastProgress

    if progress:
        num_rows = progress.get("numInputRows") or 0
        duration_ms = (progress.get("durationMs") or {}).get("triggerExecution") or 0

        event_time = progress.get("eventTime") or {}
        watermark = event_time.get("watermark") if event_time else None
        watermark = watermark or "N/A"

        print(f"[Batch {i+1}]")
        print(f"  Rows processed : {num_rows:,}")
        print(f"  Duration       : {duration_ms}ms")
        print(f"  Watermark at   : {watermark}")

        try:
            live = spark.read.format("delta").load(STREAM_OUTPUT)
            count_ = live.count()
            print(f"  Anomalies so far: {count_:,}")
        except:
            print("  Output table not yet created...")
    else:
        print(f"[Batch {i+1}] Waiting for first batch...")

    print()

Monitoring stream...

[Batch 1] Waiting for first batch...

[Batch 2] Waiting for first batch...

[Batch 3] Waiting for first batch...

[Batch 4]
  Rows processed : 0
  Duration       : 34957ms
  Watermark at   : 1995-07-28T17:22:25.000Z
  Anomalies so far: 2,103

[Batch 5]
  Rows processed : 0
  Duration       : 10765ms
  Watermark at   : 1995-07-28T17:22:25.000Z
  Anomalies so far: 2,103

[Batch 6]
  Rows processed : 0
  Duration       : 10322ms
  Watermark at   : 1995-07-28T17:22:25.000Z
  Anomalies so far: 2,103



In [0]:
results = spark.read.format("delta").load(STREAM_OUTPUT)

print(f"Total anomalies detected: {results.count():,}")
print()

print("By anomaly type:")
results.groupBy("anomaly_type").count().orderBy("count", ascending=False).show()

print("By severity:")
results.groupBy("severity").count().orderBy("severity").show()

print("\nSample — CRITICAL anomalies:")
(results
    .filter(col("severity") == "CRITICAL")
    .select("host", "window_start", "live_requests",
            "rate_multiplier", "global_z_score", "anomaly_type")
    .orderBy("rate_multiplier", ascending=False)
    .show(10, truncate=False))

Total anomalies detected: 2,103

By anomaly type:
+-------------------+-----+
|       anomaly_type|count|
+-------------------+-----+
|   LIVE_ERROR_SURGE| 1664|
|SUSPICIOUS_ACTIVITY|  435|
|     PERSONAL_SPIKE|    4|
+-------------------+-----+

By severity:
+--------+-----+
|severity|count|
+--------+-----+
|CRITICAL|  910|
|    HIGH|   13|
|  MEDIUM| 1180|
+--------+-----+


Sample — CRITICAL anomalies:
+--------------------------+-------------------+-------------+---------------+--------------+----------------+
|host                      |window_start       |live_requests|rate_multiplier|global_z_score|anomaly_type    |
+--------------------------+-------------------+-------------+---------------+--------------+----------------+
|kss2nn.nn.mobitel.telia.se|1995-07-20 15:40:00|26           |24.38          |-1.6          |PERSONAL_SPIKE  |
|kss2nn.nn.mobitel.telia.se|1995-07-20 15:45:00|18           |16.88          |-1.63         |PERSONAL_SPIKE  |
|163.205.1.45              |1995-07

In [0]:
print("=" * 55)
print("         PIPELINE COMPLETE — FINAL SUMMARY")
print("=" * 55)

layers = {
    "Bronze"     : BRONZE_PATH,
    "Silver"     : SILVER_PATH,
    "Gold/hourly_stats"   : f"{GOLD_PATH}/hourly_stats",
    "Gold/ip_behaviour"   : f"{GOLD_PATH}/ip_behaviour",
    "Gold/endpoint_stats" : f"{GOLD_PATH}/endpoint_stats",
    "Gold/anomalies"      : f"{GOLD_PATH}/../anomalies",
    "Stream output"       : STREAM_OUTPUT,
}

for name, path in layers.items():
    try:
        df = spark.read.format("delta").load(path)
        print(f"  {name:<30} {df.count():>10,} rows")
    except Exception as e:
        print(f"  {name:<30} ERROR: {e}")

         PIPELINE COMPLETE — FINAL SUMMARY
  Bronze                          1,891,715 rows
  Silver                          1,886,836 rows
  Gold/hourly_stats                     662 rows
  Gold/ip_behaviour                  81,892 rows
  Gold/endpoint_stats                21,079 rows
  Gold/anomalies                      3,613 rows
  Stream output                       2,103 rows
